# Hand2Robot 2.0 — old vs new, on Colab

Phases 1–2 of `docs/HAND2ROBOT_2.md`: run the **real** GPU models on **our** clips and compare them with the pipeline as it is today. Nothing here touches the live servers.

Three ways of seeing the hand, on every clip:

| | hand | distances |
|---|---|---|
| **A — today** | MediaPipe | guessed lens, table from the hand |
| **B** | MediaPipe | MoGe-2 depth: real table, hand's real size |
| **C** | WiLoR | MoGe-2 depth |

**Before you start (once):**
1. Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100 faster).
2. In Google Drive, make a folder **`hand2robot2`** with:
   - `pipeline.zip` — this repository on branch `pipeline-optimization` (`git archive -o pipeline.zip pipeline-optimization` in the pipeline folder),
   - `clips/` — the test clips (`.mp4`), each with its `.mp4.camera.json` beside it if the new app recorded it,
   - `truth.csv` — optional: `clip,object_width_cm` (the object's width, measured with a ruler).
3. Run the cells in order. Results are written to `hand2robot2/results/` in Drive, and the last cell prints the table to send back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/hand2robot2'
import os; print(os.listdir(ROOT))

In [ ]:
# The pipeline, as committed on the branch, and its packages — the same pinned
# install as the image (scripts/install_perception.sh), so what is measured
# here is what runs on the DGX.
!rm -rf /content/pipeline && mkdir -p /content/pipeline && cd /content/pipeline && unzip -q $ROOT/pipeline.zip
%cd /content/pipeline
!pip install -q -e ".[ml]"
!sh scripts/install_perception.sh depth wilor
!python scripts/download_retarget_weights.py --dir ./models
!python scripts/download_perception_weights.py --dir ./models depth wilor

In [ ]:
# Does each model load and run here, how fast, and how much GPU memory?
import os
os.environ.update({'PIPELINE_GPU': 'auto', 'RETARGET_MODELS_DIR': './models',
                   'RETARGET_DEPTH_MODEL': 'moge-2-vitb', 'RETARGET_HAND_MODEL': 'wilor',
                   'GPU_MEMORY_FRACTION': '0.9'})
!PIPELINE_GPU=auto RETARGET_MODELS_DIR=./models RETARGET_DEPTH_MODEL=moge-2-vitb RETARGET_HAND_MODEL=wilor GPU_MEMORY_FRACTION=0.9 python scripts/gpu_smoke.py

In [ ]:
import csv, json, logging, time, traceback
from pathlib import Path

logging.basicConfig(level=logging.WARNING)
from dfpipeline.config import settings
from dfpipeline.retarget import scene, hand_wilor
from dfpipeline.retarget.pipeline import run_hand_stage, run_robot_stage
from dfpipeline.retarget import service as rs

CONFIGS = {
    'A_today':        dict(retarget_hand_model='mediapipe', retarget_depth_model='off'),
    'B_depth':        dict(retarget_hand_model='mediapipe', retarget_depth_model='moge-2-vitb'),
    'C_wilor_depth':  dict(retarget_hand_model='wilor',     retarget_depth_model='moge-2-vitb'),
}
clips = sorted(Path(ROOT, 'clips').glob('*.mp4'))
truth = {}
if Path(ROOT, 'truth.csv').exists():
    truth = {r['clip']: float(r['object_width_cm']) for r in csv.DictReader(open(Path(ROOT, 'truth.csv')))}
out_root = Path(ROOT, 'results'); out_root.mkdir(exist_ok=True)
print(len(clips), 'clips;', len(truth), 'with a measured object width')

In [ ]:
def one(clip, name, cfg, hands):
    for k, v in cfg.items():
        setattr(settings, k, v)
    scene.depth_model.cache_clear()
    out = out_root / name / clip.stem
    if out.exists():
        import shutil; shutil.rmtree(out)
    cam_file = Path(str(clip) + '.camera.json')
    camera = json.loads(cam_file.read_text()) if cam_file.exists() else None
    t = time.time()
    code = run_hand_stage(clip, out / 'hand', hands=hands, camera=camera)
    row = {'clip': clip.name, 'config': name, 'hand_ok': code == 0, 'hand_s': round(time.time() - t, 1)}
    if code != 0:
        return row
    bundle = json.loads((out / 'hand' / 'hand_stage.json').read_text())
    row.update(rs.read_hand_metrics(out / 'hand'))
    row.update({f'p_{k}': v for k, v in (bundle.get('perception') or {}).items() if k != 'notes'})
    row['table_distance_m'] = bundle.get('table_distance_m')
    row['object_width_cm'] = round(100 * (bundle.get('object') or {}).get('diameter_m', float('nan')), 1) \
        if bundle.get('object') else None
    row['truth_width_cm'] = truth.get(clip.name)
    t = time.time()
    rcode = run_robot_stage(out / 'hand', out / 'robot')
    row['robot_ok'] = rcode == 0
    row['robot_s'] = round(time.time() - t, 1)
    if rcode == 0:
        m = rs.read_metrics(out / 'robot')
        row.update({k: m.get(k) for k in ('reachable_fraction', 'position_error_median_m', 'slowdown')})
    return row

HANDS = int(os.environ.get('HANDS', '1'))   # 2 for a two-hand task
rows = []
for clip in clips:
    for name, cfg in CONFIGS.items():
        try:
            rows.append(one(clip, name, cfg, HANDS))
        except Exception:
            rows.append({'clip': clip.name, 'config': name, 'error': traceback.format_exc()[-400:]})
        print(rows[-1].get('clip'), rows[-1].get('config'), rows[-1].get('hand_s'), rows[-1].get('error', ''))
json.dump(rows, open(out_root / 'rows.json', 'w'), indent=1, default=str)

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
cols = ['clip', 'config', 'hand_ok', 'hand_s', 'p_lens_source', 'p_hfov_deg', 'p_hand_scale', 'p_hand_scale_spread',
        'p_table_up_angle_deg', 'table_distance_m', 'object_width_cm', 'truth_width_cm',
        'hand_coverage', 'robot_ok', 'reachable_fraction', 'position_error_median_m', 'slowdown']
table = df[[c for c in cols if c in df.columns]]
if 'truth_width_cm' in table and table['truth_width_cm'].notna().any():
    table = table.assign(width_error_cm=(table['object_width_cm'] - table['truth_width_cm']).abs())
    print(table.groupby('config')['width_error_cm'].median().rename('median object-width error, cm'))
print(table.groupby('config')[['hand_s', 'hand_coverage', 'reachable_fraction', 'position_error_median_m']].median())
table.to_csv(out_root / 'summary.csv', index=False)
table

**Send back** `results/summary.csv` (and `rows.json` if anything failed). Phase 2's go/no-go: B or C clearly better than A — the object's width closer to the ruler, the arm reaching more of the path — and fast enough per clip for a night's clips.